# P-wave-earthquake-localisation-toy-box exemple notebook

In [1]:
import numpy as np

In [2]:
import generate_eq
import graphics
import commons

In [3]:
import Gauss_Newton
import Levenberg_Marquardt
import Uniq_searcher

In [4]:

# Deffine the stations, world box and V-P
#                        X       Y     Z    t
stations = np.array([[   0.0,  700.0,  0.0, 0.0],  # N°1
                     [ 700.0, 1000.0,  0.0, 0.0],  # N°2
                     [1000.0,  300.0,  0.0, 0.0],  # N°3
                     [ 500.0,  500.0, 10.0, 0.0],  # N°4
                     [ 400.0,    0.0,  0.0, 0.0]]) # N°5

box_gen = np.array([[    0.0,  1000.0],
                    [    0.0,  1000.0],
                    [-1000.0,     0.0]])

vp = 4000.0 # m/s

# Generate a random event without any noise
event, event_arr, stations, stations_true = generate_eq.generate_event(stations, box_gen, vp, 0.0)
print(f'Event before normalisation:\n\t{event}\n')

# Normalise the stations
stations_norm, event_norm = commons.centrering_arr(stations, event_arr)
event_norm_dict = {'X':float(event_norm[0]), 'Y':float(event_norm[1]),
                   'Z':float(event_norm[2]), 't':float(event_norm[3])}

print(f'Event after normalisation:\n\t{event_norm_dict}\n')

# Compute the normalised world box
limites = np.column_stack((np.min(stations_norm, axis=0), np.max(stations_norm, axis=0)))

# Set the initialisation to be the same for all iterative solver
event_ini = np.random.rand(4)
event_ini = limites[:, 0] + event_ini * (limites[:, 1]-limites[:, 0])
event_ini[2] = min(event_ini[2], -1)
event_ini_dict = {'X':float(event_ini[0]), 'Y':float(event_ini[1]),
                  'Z':float(event_ini[2]), 't':float(event_ini[3])}

print(f'Initialisation:\n\t{event_ini_dict}\n')


Event before normalisation:
	{'X': 400.5654020281135, 'Y': 726.9254183053072, 'Z': -925.4550553797676, 't': 1785163982.9667}

Event after normalisation:
	{'X': -99.43459797188649, 'Y': 226.92541830530718, 'Z': -935.4550553797676, 't': -0.241926908493042}

Initialisation:
	{'X': 152.47372053209233, 'Y': -363.81740122902573, 'Z': -6.4651046506534415, 't': 0.02852619761333747}



In [5]:
event_best_GN, history_GN, cost_story_GN, misfit_fin_GN = Gauss_Newton.Gauss_Newton(
    stations=stations_norm,
    event_test=np.copy(event_ini),
    n_iteration=1_000,
    steps=np.array([1.0, 1.0, 1.0, 0.0005]),
    vp=vp)

print('Best fitt with:')
print(f'\t- RMSE = {misfit_fin_GN}')
print(f'\t- Event = {event_best_GN}')

_ = graphics.history_2d(cost_story_GN, 'RMSE', log_y=True)
_ = graphics.history_2d(history_GN[:, 3], 'Timing')
_ = graphics.history_3d(history_GN[:, :3], stations_norm)

Best fitt with:
	- RMSE = 1.1037189987442446e-08
	- Event = {'X': -99.43489704099824, 'Y': 226.92518244336748, 'Z': -935.4558233179968, 't': -0.24192714892801825}


In [6]:
event_best_LM, history_LM, cost_story_LM, misfit_fin_LM = Levenberg_Marquardt.Levenberg_Marquardt(
    stations=stations_norm,
    event_test=np.copy(event_ini),
    n_iteration=1_000,
    vp=vp)

print('Best fitt with:')
print(f'\t- RMSE = {misfit_fin_LM}')
print(f'\t- Event = {event_best_LM}')

_ = graphics.history_2d(cost_story_LM, 'RMSE', log_y=True)
_ = graphics.history_2d(history_LM[:, 3], 'Timing')
_ = graphics.history_3d(history_LM[:, :3], stations_norm)

Best fitt with:
	- RMSE = 1.1037190001703776e-08
	- Event = {'X': -99.43489704099822, 'Y': 226.92518244336722, 'Z': -935.4558233179962, 't': -0.24192714892801812}


In [7]:
event_best_US, history_US, cost_story_US, misfit_fin_US = Uniq_searcher.gradient_descent(
    stations=stations_norm,
    event_test=np.copy(event_ini),
    n_iteration=200_000,
    ranX=1,
    ranY=1,
    ranZ=1,
    rant=0.0005,
    lr_decay=0.05,
    patience=500,
    vp=4000.0)

print('Best fitt with:')
print(f'\t- RMSE = {misfit_fin_US}')
print(f'\t- Event = {event_best_US}')

_ = graphics.history_2d(cost_story_US, 'RMSE', log_y=True)
_ = graphics.history_2d(history_US[:, 3], 'Timing')
_ = graphics.history_3d(history_US[:, :3], stations_norm)

Best fitt with:
	- RMSE = 1.1037209194356053e-08
	- Event = {'X': -99.43489690980049, 'Y': 226.92518215858877, 'Z': -935.4558216718922, 't': -0.2419271485512676}
